# 04-Checkpointer 与 State 管理

## 持久化记忆：Checkpointer 与 State 管理

InMemorySaver · thread_id · get_state · 时间旅行 · 第四课 · 约 40 分钟

**上节课**我们学会了条件路由——让图根据状态动态选路。

**这节课**我们学如何让图"记住"状态——每次调用之间自动保存和恢复 State。

### 目录

-   为什么需要记忆？
-   什么是 Checkpointer？
-   开启记忆：InMemorySaver + thread_id
-   State 的查看与修改：get_state / update_state
-   持久化方案对比
-   时间旅行：get_state_history
-   Demo 1：跨轮对话（累加器）
-   Demo 2：编辑对话历史
-   Demo 3：时间旅行（重放与分叉）
-   常见坑
-   总结

### 一、为什么需要记忆？

到目前为止，我们的图都是"一次性"的——每次 `invoke()` 都从初始 State 开始，跑完就结束。但实际场景需要跨轮次记忆：

-   **对话机器人：** "我叫小明" → "我今年 25 岁" → "我叫什么名字？" → 需要记得之前说过的内容
-   **累加器模式：** 每次调用往列表里添加一项，下次调用列表里应该包含之前的所有项
-   **多步骤流程：** 第一步收集信息，第二步基于第一步的结果继续

> **类比：便签纸 vs 笔记本**
> 
> **无记忆：** 每次都在一张新便签纸上写东西。上一次写了什么？不知道——纸已经被扔掉了。
> 
> **有记忆：** 用一本笔记本，每次翻到上次写的那页继续写。上次的内容还在，可以随时回看。
> 
> Checkpointer 就是这本"笔记本"——它自动保存每次的 State，下次接着用。

### 二、什么是 Checkpointer？

Checkpointer（检查点）是 LangGraph 的**持久化层**。它的工作很简单：

1.  图的每一步执行完后，自动把当前 State **保存**到 Checkpointer
2.  下一次 `invoke()` 时，从 Checkpointer **加载**上次保存的 State



In [23]:
# 这段用于“行为示意”，避免打断 notebook 的顺序执行。
# 无记忆：每次都是全新的 State
# graph.invoke(initial_state)     # State 从头开始
# graph.invoke(initial_state)     # 又是从头开始，和第一次没关系
#
# 有记忆：State 在调用之间自动积累
# graph.invoke(input1, config)    # State = {初始} + {input1}
# graph.invoke(input2, config)    # State = {初始} + {input1} + {input2}


> **关键理解：**Checkpointer 不是"记忆"的开关——它是图的**运行时基础设施**。除了跨轮记忆外，它还是 Interrupt（下一课）、时间旅行（本课）、以及状态回滚等高级功能的基础。

**为什么一个"存档系统"能撑起这么多功能？**

关键在于：挂上 Checkpointer 后，图**每执行完一步，就把当时的完整 State 拍一张快照**存进存档柜。跑完 A → B → C，柜子里实际躺着一串快照：

```text
快照0: steps=[],            next=('__start__',)   ← 输入刚进来
快照1: steps=[],            next=('step_a',)
快照2: steps=['A'],         next=('step_b',)
快照3: steps=['A','B'],     next=('step_c',)
快照4: steps=['A','B','C'], next=()              ← 跑完了
```

注意每张快照不仅有**当时的数据**（values），还有 **「下一步该干什么」**（next）——所以快照是可以「接着跑」的，不只是用来看的。于是，四个看似不相关的功能，其实只是对同一串快照的**四种读写姿势**：

<figure style="text-align:center;">
  <img src="../assets/04_checkpointer_cabinet.png" width="800" />
  <figcaption style="text-align:center; margin-top: 6px;">Checkpointer存放柜.png</figcaption>
</figure>

| 功能  | 对存档的操作 | 对应 API |
| --- | --- | --- |
| ① 跨轮记忆 | 读最新快照，叠加新输入继续跑 | invoke(input, config) |
| ② Interrupt（下一课） | 暂停瞬间存档退出；之后读档继续 | interrupt() + Command(resume=...) |
| ③ 时间旅行 | 读任意旧快照，原样重跑 | get_state_history() + invoke(None, past.config) |
| ④ 状态回滚/分叉 | 读旧快照，先改一笔再往下跑 | update_state(past.config, ...) |

> **一句话心智模型**
> 
> **Checkpointer = 自动按帧存档的游戏机。**
> 
> 记忆 = 读最新档接着玩；Interrupt = 存档退出、改天读档；时间旅行 = 翻旧档重玩；回滚 = 读旧档改个操作再玩。
> 
> 四个功能没有四套机制——**只有一套存档，四种读写姿势**。你以为买的是个笔记本，实际上买的是一台带全程录像的时光机，「记忆」只是它最朴素的用法。

### 三、开启记忆：InMemorySaver + thread_id

只需要两步就能让图"记住"状态：

**Step 1：创建 Checkpointer**



In [24]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()


**Step 2：编译时传入**



In [25]:
# 编译时把 checkpointer 挂上去（示意）
# graph = builder.compile(checkpointer=memory)   # ← 就这一行，其他代码不变


**Step 3：运行时指定 thread_id**

每次 `invoke()` 时必须传入 `config`，用 `thread_id` 标识"这是谁"：



In [26]:
# 运行时指定 thread_id（示意）
# config = {"configurable": {"thread_id": "session_1"}}
# result = graph.invoke(inputs, config=config)


> **thread_id 的类比：游戏存档位**
> 
> **thread_id = 存档位编号**
> 
> -   用 `"thread_1"` 玩 → 存档在 1 号位 → 下次用 `"thread_1"` 继续
> -   用 `"thread_2"` 玩 → 新开一个存档 → 和 `"thread_1"` 互不干扰
> -   同一个 thread_id 的多次 invoke = 同一个存档位的多次读写

**完整的最小示例**



In [27]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    messages: Annotated[list[str], operator.add]

def chat(state: State):
    msg = state["messages"][-1]
    reply = f"你说了：{msg}"
    return {"messages": [reply]}

# 编译——传入 Checkpointer
memory = InMemorySaver()
graph = StateGraph(State).add_node("chat", chat)
graph.add_edge(START, "chat")
graph.add_edge("chat", END)
graph = graph.compile(checkpointer=memory)

# 第 1 轮
config = {"configurable": {"thread_id": "alice"}}
r1 = graph.invoke({"messages": ["你好"]}, config)
print(r1["messages"])
# -> ['你好', '你说了：你好']

# 第 2 轮——用同一个 thread_id！
r2 = graph.invoke({"messages": ["还记得我之前说了什么吗？"]}, config)
print(r2["messages"])
# -> ['你好', '你说了：你好', '还记得我之前说了什么吗？', '你说了：还记得我之前说了什么吗？']
#     ↑ 之前的消息还在！

['你好', '你说了：你好']
['你好', '你说了：你好', '还记得我之前说了什么吗？', '你说了：还记得我之前说了什么吗？']



> **注意：** 这里用了 `operator.add` reducer（第二课内容），所以 messages 列表在每次 invoke 时自动追加，而不是被覆盖。Checkpointer + Reducer 结合，就形成了"记忆"。

> **❓ 检查理解 ①**
> 
> 挂了 Checkpointer 的图，用 `thread_id="a"` 聊了 3 轮后，改用 `thread_id="b"` invoke 一次。此时 "b" 能看到 "a" 的历史吗？
> 
> -   A. 能，Checkpointer 是全局共享的
> -   B. 不能，不同 thread_id 的 State 完全隔离
> -   C. 能看到最后一轮，看不到更早的

> **✅ 答案：B**
> 
> thread_id 就是存档位编号——不同存档位完全隔离。这正是多用户服务的基础：每个用户一个 thread_id，互不串台。

### 四、State 的查看与修改：get_state / update_state

Checkpointer 开启后，你可以随时**查看**和**修改**图中的 State：

**get_state：查看当前 State**

查看某个 thread 的最新 State：



In [28]:
snapshot = graph.get_state(config)
print(snapshot.values)     # 当前 State 的值
print(snapshot.next)       # 下一步要执行的节点（元组，空=已结束）
print(snapshot.tasks)      # 当前待执行的任务（中断时有用）

{'messages': ['你好', '你说了：你好', '还记得我之前说了什么吗？', '你说了：还记得我之前说了什么吗？']}
()
()



**update_state：手动修改 State**

向某个 thread 注入新的 State 值，就像"替某个节点写入了结果"：



In [29]:
# 修改 State：假装是 "chat" 节点写入了新消息
graph.update_state(
    config,
    {"messages": ["这是手动注入的消息"]},
    as_node="chat"      # 以 chat 节点的身份写入
)

{'configurable': {'thread_id': 'alice',
  'checkpoint_ns': '',
  'checkpoint_id': '1f191941-031b-63bf-8005-bb94776e4936'}}


> **as_node 参数：** 告诉 Checkpointer"这个消息是由哪个节点产生的"。如果不传，默认走当前图的入口节点。这在需要"人工修正"时非常有用——你可以修正 LLM 的错误输出。

### 五、持久化方案对比

LangGraph 支持多种 Checkpointer，从开发测试到生产环境：

| 方案  | 导入路径 | 持久化 | 适用场景 |
| --- | --- | --- | --- |
| InMemorySaver (别名 MemorySaver) | langgraph.checkpoint.memory | ❌ 内存（进程重启丢失） | 开发测试 |
| SqliteSaver | langgraph.checkpoint.sqlite | ✅ 磁盘文件 | 本地应用/单机部署 |
| PostgresSaver | langgraph.checkpoint.postgres | ✅ 数据库 | 生产环境/多机部署 |

**SqliteSaver 示例（持久化到磁盘）**



In [30]:
# SqliteSaver（示意）
#
# import sqlite3
# from langgraph.checkpoint.sqlite import SqliteSaver
#
# conn = sqlite3.connect("my_graph.db", check_same_thread=False)
# disk_saver = SqliteSaver(conn)
# graph = builder.compile(checkpointer=disk_saver)
#
# 之后用法和 InMemorySaver 完全一样。
# 即使重启 Python 进程，State 也能从 my_graph.db 中恢复。


> **开发建议：** 开发时用 `InMemorySaver`，需要持久化时用 `SqliteSaver`。API 完全一样，只需改导入和初始化那一行。

### 六、时间旅行：get_state_history

既然 Checkpointer 保存了每一步的 State 快照，那自然可以回到任意一个历史状态。

**查看历史**



In [31]:
# 获取所有历史快照（最新在前，最后一个是 __start__ 起点）
all_states = list(graph.get_state_history(config))
print(f"共有 {len(all_states)} 个历史状态")

# 查看倒数第二个状态的详细内容
past = all_states[-2]
print(past.values)     # 当时的 State
print(past.next)       # 当时下一步要执行的节点
print(past.config)     # 包含 checkpoint_id，可用于重放

共有 7 个历史状态
{'messages': ['你好']}
('chat',)
{'configurable': {'thread_id': 'alice', 'checkpoint_ns': '', 'checkpoint_id': '1f191941-02a7-6840-8000-ae3e6bfa2a6d'}}



**重放（Replay）**

从某个历史点重新执行，**输入完全一样**：



In [32]:
past = all_states[-2]                    # 选一个历史检查点
for event in graph.stream(None, past.config):
    print(event)

{'chat': {'messages': ['你说了：你好']}}



**分叉（Fork）**

从某个历史点开始，但**修改输入走一条新路**——像游戏读档后选不同选项：



In [33]:
past = all_states[-2]                     # 选一个历史检查点

# 修改历史状态，创建一个新的分叉
new_config = graph.update_state(
    past.config,
    {"messages": ["修改后的消息"]}
)

# 从分叉点重新执行
for event in graph.stream(None, new_config):
    print(event)

{'chat': {'messages': ['你说了：修改后的消息']}}



> **分叉时注意 Reducer 规则（实测）：**`update_state` 写入的值会按字段的 Reducer 处理——带 `operator.add` 的列表字段是**追加**，不是替换。想"改写历史"而非"追加历史"，字段要用默认覆盖语义，或在 Reducer 里自行设计替换逻辑（如第二课 `add_messages` 的按 ID 替换）。

> **时间旅行的类比：游戏存档**
> 
> **查看历史** = 列出所有存档位
> 
> **重放** = 载入存档，按同样的操作再玩一遍
> 
> **分叉** = 载入存档，然后选一条不同的路
> 
> 时间旅行在调试中极为有用——你可以回到 LLM 犯错之前的状态，手动修正后继续。

### 七、Demo 1：跨轮对话（累加器）

验证 Checkpointer 如何让 State 在多次 invoke 之间积累：

In [34]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    history: Annotated[list[str], operator.add]

def respond(state: State):
    last = state["history"][-1]
    reply = f"[回] {last}"
    print(f"  回复：{reply}")
    return {"history": [reply]}

builder = StateGraph(State)
builder.add_node("respond", respond)
builder.add_edge(START, "respond")
builder.add_edge("respond", END)

memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

# 用同一个 thread_id 进行 3 轮对话
config = {"configurable": {"thread_id": "demo1"}}

for i, msg in enumerate(["你好", "我叫小明", "我叫什么？"]):
    print(f"第 {i+1} 轮 输入：{msg}")
    result = graph.invoke({"history": [msg]}, config)
    print(f"  State 累计 {len(result['history'])} 条:\n")

print("=== 最终 State ===")
for item in result["history"]:
    print(f"  {item}")

print("\n=== 用新 thread_id 测试隔离性 ===")
config2 = {"configurable": {"thread_id": "demo1_new"}}
result2 = graph.invoke({"history": ["新对话"]}, config2)
print(f"新对话有 {len(result2['history'])} 条（只有输入）")
print(f"旧对话有 {len(result['history'])} 条（互不干扰）")

第 1 轮 输入：你好
  回复：[回] 你好
  State 累计 2 条:

第 2 轮 输入：我叫小明
  回复：[回] 我叫小明
  State 累计 4 条:

第 3 轮 输入：我叫什么？
  回复：[回] 我叫什么？
  State 累计 6 条:

=== 最终 State ===
  你好
  [回] 你好
  我叫小明
  [回] 我叫小明
  我叫什么？
  [回] 我叫什么？

=== 用新 thread_id 测试隔离性 ===
  回复：[回] 新对话
新对话有 2 条（只有输入）
旧对话有 6 条（互不干扰）



运行后：

```text
第 1 轮 输入：你好
  回复：[回] 你好
  State 累计 2 条:

第 2 轮 输入：我叫小明
  回复：[回] 我叫小明
  State 累计 4 条:

第 3 轮 输入：我叫什么？
  回复：[回] 我叫什么？
  State 累计 6 条:

=== 最终 State ===
  你好
  [回] 你好
  我叫小明
  [回] 我叫小明
  我叫什么？
  [回] 我叫什么？

=== 用新 thread_id 测试隔离性 ===
新对话有 2 条（只有输入）
旧对话有 6 条（互不干扰）
```

> **关键观察：** 同一个 thread_id 的多次 invoke 自动累积历史；不同的 thread_id 完全隔离。这就是 Checkpointer 提供的最基础能力——跨轮次记忆。

### 八、Demo 2：编辑对话历史

通过 `get_state` 查看正在进行的对话，用 `update_state` 手动修正：

In [35]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AIMessage, HumanMessage

class State(TypedDict):
    messages: Annotated[list, operator.add]

def chat(state: State):
    last_msg = state["messages"][-1]
    if isinstance(last_msg, HumanMessage):
        reply = AIMessage(content=f"你说了：{last_msg.content}")
        return {"messages": [reply]}
    return {}

builder = StateGraph(State)
builder.add_node("chat", chat)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "edit_demo"}}

# 第 1 轮：正常对话
graph.invoke(
    {"messages": [HumanMessage(content="我的名字是张三")]},
    config
)

# 查看当前 State
snapshot = graph.get_state(config)
print("当前消息数：", len(snapshot.values["messages"]))
print("最后一条：", snapshot.values["messages"][-1].content)

# 手动修改 State：注入一条修正消息
print("\n--- 注入人工反馈 ---\n")
graph.update_state(
    config,
    {"messages": [AIMessage(content="你叫张三，我记住了。")]},
    as_node="chat"
)

# 第 2 轮：验证修正效果
result = graph.invoke(
    {"messages": [HumanMessage(content="我叫什么？")]},
    config
)
print("\n所有消息：")
for m in result["messages"]:
    role = "H" if isinstance(m, HumanMessage) else "A"
    print(f"  [{role}] {m.content}")

当前消息数： 2
最后一条： 你说了：我的名字是张三

--- 注入人工反馈 ---


所有消息：
  [H] 我的名字是张三
  [A] 你说了：我的名字是张三
  [A] 你叫张三，我记住了。
  [H] 我叫什么？
  [A] 你说了：我叫什么？



运行后：

```text
当前消息数：2
最后一条：你说了：我的名字是张三

--- 注入人工反馈 ---

所有消息：
  [H] 我的名字是张三
  [A] 你说了：我的名字是张三
  [A] 你叫张三，我记住了。 <-- 手动注入的
  [H] 我叫什么？
  [A] 你说了：我叫什么？
```

> **为什么这有用？**在实际的 LLM 应用中，AI 的回答可能不准确或偏离方向。通过 `update_state`，人可以**在 AI 回答之后插入修正**，后续的对话会基于修正后的 State 继续——这是 HITL（下节课）的一种轻量形式。

### 九、Demo 3：时间旅行（重放与分叉）

用 `get_state_history` 回到过去，重新"选择"一条不同的路：

In [36]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    steps: Annotated[list[str], operator.add]

def step_a(state: State):
    return {"steps": ["A"]}

def step_b(state: State):
    return {"steps": ["B"]}

def step_c(state: State):
    return {"steps": ["C"]}

builder = StateGraph(State)
builder.add_node("step_a", step_a)
builder.add_node("step_b", step_b)
builder.add_node("step_c", step_c)
builder.add_edge(START, "step_a")
builder.add_edge("step_a", "step_b")
builder.add_edge("step_b", "step_c")
builder.add_edge("step_c", END)

memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "travel"}}

# 第 1 次运行：A → B → C
result = graph.invoke({"steps": []}, config)
print("原始执行：", result["steps"])

# 查看所有历史状态
all_states = list(graph.get_state_history(config))
print(f"\n共有 {len(all_states)} 个历史检查点：")
for s in all_states:
    print(f"  steps={s.values['steps']}, next={s.next}")

# 回到执行 step_b 之前的状态：用 next 精确定位，比数下标更可靠
before_b = next(s for s in all_states if s.next == ("step_b",))
print(f"\n回到过去：steps={before_b.values['steps']}，即将执行 {before_b.next}")

# 分叉：修改 State，走不同的路
new_config = graph.update_state(
    before_b.config,
    {"steps": ["A_X"]}    # 注意：add Reducer → 这是【追加】"A_X"，不是替换"A"
)

# 从分叉点重新执行
result2 = graph.invoke(None, new_config)
print("分叉后执行：", result2["steps"])

原始执行： ['A', 'B', 'C']

共有 5 个历史检查点：
  steps=['A', 'B', 'C'], next=()
  steps=['A', 'B'], next=('step_c',)
  steps=['A'], next=('step_b',)
  steps=[], next=('step_a',)
  steps=[], next=('__start__',)

回到过去：steps=['A']，即将执行 ('step_b',)
分叉后执行： ['A', 'A_X', 'B', 'C']



运行后：

```text
原始执行：['A', 'B', 'C']

共有 5 个历史检查点：
  steps=['A', 'B', 'C'], next=()
  steps=['A', 'B'], next=('step_c',)
  steps=['A'], next=('step_b',)
  steps=[], next=('step_a',)
  steps=[], next=('__start__',)

回到过去：steps=['A']，即将执行 ('step_b',)
分叉后执行：['A', 'A_X', 'B', 'C']
```

**两个关键观察（实测）：**

-   **检查点是 5 个不是 4 个**——历史里还有一个 `next=('__start__',)` 的「输入刚进来、还没跑」的起点快照。这也是为什么定位历史点时，按 `s.next == ("step_b",)` 匹配比数下标 `[-2]` 更可靠。
-   **分叉结果是 `['A', 'A_X', 'B', 'C']`**——「A」还在！因为 `steps` 字段挂了 `operator.add`，`update_state` 写入的 `["A_X"]` 被**追加**而非替换（上一节的 note 说的就是这件事）。带 Reducer 的字段，「改写历史」实际是「在历史上加一笔」。

> **时间旅行的价值：** 调试时你可以"回到" LLM 回答之前，修改 Prompt 或上下文，重新执行看不同结果。在对话 AI 中，如果 LLM 在某轮回答错了，你可以回到那一轮之前修正，而不是从头开始。

> **❓ 检查理解 ②**
> 
> 「重放」和「分叉」的本质区别是什么？
> 
> -   A. 重放用 `past.config` 原样执行；分叉先 `update_state` 改历史，再从新 config 执行
> -   B. 重放更快，分叉更慢
> -   C. 重放只能看不能跑，分叉才会真正执行

> **✅ 答案：A**
> 
> 一句话：重放 = 读档照原样再玩；分叉 = 读档后先改装备再玩。update_state 返回的新 config 指向一条新的历史分支，原历史不受影响。

### 十、常见坑

> **坑 1：有 Checkpointer 但没传 config**
> 
> `invoke()` 时如果不传 config（缺 thread_id），Checkpointer 不会生效：
> 
> 🔴 `graph.invoke(inputs) # 没传 config，无记忆`
> 
> ✅ `graph.invoke(inputs, config={"configurable": {"thread_id": "x"}})`

> **坑 2：Reducer 类型没设置对**
> 
> 如果 State 字段没有合适的 Reducer（如缺少 `operator.add`），多次 invoke 不会累积，而是直接覆盖。记忆失效。
> 
> ✅ 需要累积的字段用 `Annotated[list[str], operator.add]`

> **坑 3：MemorySaver vs InMemorySaver**
> 
> 实测两者**是同一个类**（`MemorySaver is InMemorySaver == True`）。官方文档现行用名是 `InMemorySaver`，`MemorySaver` 是保留的旧别名——和旧教程对不上时不用慌，写哪个都能跑。本课代码与官方文档保持一致，用 `InMemorySaver`。

> **坑 4：update_state 后 next 节点变了**
> 
> `update_state` 不只会修改 State 值——如果操作后还有未执行的节点，`next` 会相应变化。如果需要"重置"图的状态，需要额外处理。

### 十一、总结

| 概念  | 一句话 |
| --- | --- |
| Checkpointer | 图的持久化层，每步执行后自动保存 State 快照 |
| InMemorySaver | 内存型 Checkpointer，开发测试首选 |
| SqliteSaver | 磁盘持久化 Checkpointer，进程重启可恢复 |
| thread_id | 对话/会话的唯一标识，同 ID 共享 State，不同 ID 隔离 |
| get_state | 查看指定 thread 的当前 State（values/next/tasks） |
| update_state | 手动向 thread 注入 State 更新（人工修正） |
| get_state_history | 获取线程的所有历史 State 快照 |
| 重放  | 从历史检查点重新执行（相同输入） |
| 分叉  | 从历史检查点用新输入执行（不同路径） |

> **一句话总结**
> 
> **Checkpointer = 存档系统**
> 
> 每次 invoke 自动"保存游戏"，下次用同一个 thread_id 加载存档继续玩。
> 
> get_state = 查看存档，update_state = 修改存档，get_state_history = 列出所有存档位。

> **下一课预告**
> 
> Checkpointer 就是下节课 **Human-in-the-Loop** 的基础——`interrupt()` 之所以能"暂停并等待"，正是因为 Checkpointer 保存了暂停时的 State。第五课我们让图停下来等人审批。

📖 参考：

-   [LangGraph 官方文档 - 持久化](https://docs.langchain.com/oss/python/langgraph/persistence)
-   [Interrupts 文档](https://docs.langchain.com/oss/python/langgraph/interrupts)